# Ground Truth — palier 2 (Adam)

1. **Exécution → Modifier le type d'exécution → GPU T4**, puis Enregistrer.
2. **Exécution → Tout exécuter.** Si Colab demande l'accès au secret `HF_TOKEN`, clique sur *Annuler*.
3. Laisse tourner (~1 h 15). À la fin, la dernière cellule affiche un **résumé** et télécharge `resultats.zip` : envoie les deux à Claude.

Ce que fait le notebook :
- lit les PDF (texte + OCR des scans et des graphiques) et construit 4 index (stratégies de découpage) ;
- compare les méthodes de recherche (BM25, dense, hybride, hybride + reranker) sur la validation ;
- répond aux 100 questions de validation avec le générateur officiel, les note avec le juge officiel ;
- calibre l'abstention (quand répondre INTROUVABLE) ;
- répond aux 200 questions du leaderboard v1 et prépare le fichier de soumission.

In [ ]:
import subprocess, time
T0 = time.time()
try:
    out = subprocess.check_output(["nvidia-smi", "--query-gpu=name,memory.total,memory.used",
                                   "--format=csv,noheader,nounits"], text=True).strip().splitlines()[0]
    print("GPU :", out)
except Exception:
    raise SystemExit("Pas de GPU : Exécution > Modifier le type d'exécution > GPU T4, puis Tout exécuter.")

In [ ]:
%cd /content
!test -d Ground-Truth || git clone -q https://github.com/adam2765/Ground-Truth.git
!git -C Ground-Truth pull -q
%cd /content/Ground-Truth
!apt-get -qq install -y tesseract-ocr tesseract-ocr-fra > /dev/null 2>&1 && tesseract --list-langs | tail -n +2 | tr '\n' ' '
!pip install -q -r requirements.txt sentence-transformers PyStemmer pytesseract

In [ ]:
# Corpus officiel (110 PDF v1 + 20 v2 ; les 10 scans sont générés par le script)
!python corpus/download_corpus.py | tail -n 3

## 1. Indexation
Lecture des PDF avec OCR (les pages déjà lues sont dans `cache/pages_v1.json.gz` pour gagner ~20 min ;
`--reparse` refait tout), puis 4 stratégies de découpage et leurs embeddings (bge-m3).

In [ ]:
import os, gzip, shutil
os.makedirs("index2", exist_ok=True)
if os.path.exists("cache/pages_v1.json.gz") and not os.path.exists("index2/pages.json"):
    with gzip.open("cache/pages_v1.json.gz", "rb") as f, open("index2/pages.json", "wb") as g:
        shutil.copyfileobj(f, g)
!python -m rag2.pipeline index --pdf-dir corpus/pdf --index-dir index2

## 2. Ablation de la recherche (chunking × méthode)
Choisit la meilleure combinaison (Recall@5) et l'écrit dans `index2/best.json`.

In [ ]:
!python -m rag2.ablation --index-dir index2 --questions data/validation_v1.json

## 3. Validation : réponses, notation par le juge officiel, calibration de l'abstention

In [ ]:
!python -m rag2.pipeline answer --index-dir index2 --questions data/validation_v1.json --out runs/val_raw.json --backend hf

In [ ]:
!python -m groundtruth.evaluate --pred runs/val_raw.json --gold data/validation_v1.json --judge hf --out runs/val_report.json

In [ ]:
!python -m rag2.calibrate fit --raw runs/val_raw.json --report runs/val_report.json --gold data/validation_v1.json --out index2/abstention.json
!python -m rag2.calibrate apply --raw runs/val_raw.json --rule index2/abstention.json --out runs/val_final.json

## 4. Leaderboard v1 : 200 questions, puis fichier de soumission

In [ ]:
!python -m rag2.pipeline answer --index-dir index2 --questions data/classement_v1_questions.json --out runs/cls_raw.json --backend hf
!python -m rag2.calibrate apply --raw runs/cls_raw.json --rule index2/abstention.json --out runs/classement_v1.json
!python -m groundtruth.submit --pred runs/classement_v1.json --phase v1

## 5. Résumé à envoyer à Claude

In [ ]:
import json, glob, zipfile, os
from collections import Counter
print(f"Durée totale : {(time.time() - T0) / 60:.0f} min\n")
abl = json.load(open("runs/ablation_retrieval.json"))
print("RECHERCHE (validation, questions avec réponse)")
for r in abl:
    print(f"  {r['strategy']:13s} {r['mode']:14s} R@5 {r['recall@5']:.3f}  MRR {r['mrr']:.3f}")
print("  meilleure :", json.load(open("index2/best.json")))
rep = json.load(open("runs/val_report.json"))
print("\nVALIDATION, réponses brutes (juge officiel) :", rep["score_confiance"], rep["issues"])
print("  par type :", {t: v["score"] for t, v in rep["par_type"].items()})
print("ABSTENTION CALIBRÉE :", json.load(open("index2/abstention.json")))
fails = [d for d in rep["details"] if d["points"] < 0]
print("  réponses fausses (avant calibration) :", Counter(d["type"] for d in fails))
subs = sorted(glob.glob("submissions/v1/*"))
print("\nSOUMISSION :", subs[-1] if subs else "aucune")
files = ["runs/ablation_retrieval.json", "runs/val_report.json", "runs/val_raw.json", "runs/cls_raw.json",
         "runs/classement_v1.json", "index2/abstention.json", "index2/best.json"] + subs[-1:]
with zipfile.ZipFile("resultats.zip", "w", zipfile.ZIP_DEFLATED) as z:
    for f in files:
        if os.path.exists(f):
            z.write(f)
from google.colab import files as colab_files
colab_files.download("resultats.zip")